# VSLM Models Evaluation on Kvasir-VQA-x1

**Models:** MoE-TinyMed, BiomedGPT, Florence-2, MiniGPT-4

**Metrics:** Accuracy, F1, BLEU, ROUGE-L, ECE

**Baseline:** BLIP-2 Zero-shot

## 1. Install Dependencies

**⚠️ After running this cell → Runtime → Restart runtime → skip to Cell 2.**

In [ ]:
!pip install -q datasets transformers accelerate pillow pandas tqdm
!pip install -q einops bitsandbytes sentencepiece protobuf
!pip install -q nltk rouge-score
print("\n" + "="*60)
print("  RESTART RUNTIME NOW: Runtime → Restart runtime")
print("  Then skip this cell, run from Cell 2.")
print("="*60)

## 2. Imports & Configuration
**Start here after restart.**

In [ ]:
import os, json, gc, math, re
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoProcessor,
    InstructBlipProcessor, InstructBlipForConditionalGeneration
)
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

USE_DRIVE = True
NUM_SAMPLES = 20

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector"
else:
    PROJECT_DIR = "/content/medical-vqa"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
IMAGE_DIR = os.path.join(DATA_DIR, "images")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results", "predictions")
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 64
print(f"Device: {DEVICE}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch: {torch.__version__}")
import transformers; print(f"Transformers: {transformers.__version__}")

## 3. Download Dataset (if USE_DRIVE = False)

In [ ]:
if not USE_DRIVE:
    from datasets import load_dataset
    os.makedirs(IMAGE_DIR, exist_ok=True)
    ds_host = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
    seen = set()
    for row in tqdm(ds_host, desc="Saving images"):
        if row["img_id"] not in seen:
            row["image"].save(os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg"))
            seen.add(row["img_id"])
    for split in ["train", "test"]:
        ds = load_dataset("SimulaMet/Kvasir-VQA-x1", split=split)
        records = [{"img_id": r["img_id"], "complexity": r["complexity"],
                    "question": r["question"], "answer": r["answer"],
                    "question_class": r["question_class"]} for r in ds]
        pd.DataFrame(records).to_csv(os.path.join(DATA_DIR, f"kvasir_vqa_x1_{split}.csv"), index=False)
else:
    print("[INFO] Using data from Google Drive.")

## 4. Evaluation Utilities (Accuracy, F1, BLEU, ROUGE-L, ECE)

In [ ]:
# --- Metric Functions ---
smoother = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_word_f1(pred, gt):
    p_tok = set(pred.strip().lower().split())
    g_tok = set(gt.strip().lower().split())
    if not p_tok or not g_tok: return 0.0
    common = p_tok & g_tok
    if not common: return 0.0
    prec = len(common)/len(p_tok); rec = len(common)/len(g_tok)
    return 2*prec*rec/(prec+rec)

def compute_bleu(pred, gt):
    """Sentence-level BLEU (smoothed) between prediction and ground truth."""
    ref = gt.strip().lower().split()
    hyp = pred.strip().lower().split()
    if not ref or not hyp: return 0.0
    try:
        return sentence_bleu([ref], hyp, smoothing_function=smoother)
    except Exception:
        return 0.0

def compute_rouge_l(pred, gt):
    """ROUGE-L F-measure between prediction and ground truth."""
    if not pred.strip() or not gt.strip(): return 0.0
    scores = rouge.score(gt.strip().lower(), pred.strip().lower())
    return scores['rougeL'].fmeasure

def compute_ece(confidences, accuracies, n_bins=10):
    """Expected Calibration Error.
    Since generative models don't output class probabilities,
    we use word_f1 as a proxy confidence score."""
    if not confidences: return 0.0
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total = len(confidences)
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i+1]
        mask = [(lo <= c < hi) for c in confidences]
        n_bin = sum(mask)
        if n_bin == 0: continue
        avg_conf = np.mean([c for c, m in zip(confidences, mask) if m])
        avg_acc = np.mean([a for a, m in zip(accuracies, mask) if m])
        ece += (n_bin / total) * abs(avg_acc - avg_conf)
    return ece

def select_diverse_samples(df, n_samples, seed=42):
    samples = []
    for c in sorted(df["complexity"].unique()):
        subset = df[df["complexity"] == c]
        samples.append(subset.sample(n=min(max(1, n_samples//3), len(subset)), random_state=seed))
    return pd.concat(samples).head(n_samples)

def evaluate_single(row, prediction):
    gt = str(row["answer"])
    em = prediction.strip().lower() == gt.strip().lower()
    f1 = compute_word_f1(prediction, gt)
    bleu = compute_bleu(prediction, gt)
    rouge_l = compute_rouge_l(prediction, gt)
    return {
        "img_id": row["img_id"], "complexity": int(row["complexity"]),
        "question_class": row["question_class"], "question": row["question"],
        "ground_truth": gt, "prediction": prediction,
        "exact_match": em, "word_f1": round(f1, 3),
        "bleu": round(bleu, 3), "rouge_l": round(rouge_l, 3),
    }

def compute_summary(results, model_name):
    if not results: return {"model": model_name, "error": "No results"}
    total = len(results)
    em = sum(1 for r in results if r["exact_match"])
    f1s = [r["word_f1"] for r in results]
    bleus = [r["bleu"] for r in results]
    rouges = [r["rouge_l"] for r in results]
    accuracies = [1.0 if r["exact_match"] else 0.0 for r in results]
    ece = compute_ece(f1s, accuracies)
    rdf = pd.DataFrame(results)
    per_c = {}
    for c in sorted(rdf["complexity"].unique()):
        cdf = rdf[rdf["complexity"] == c]
        per_c[f"level_{c}"] = {
            "exact_matches": int(cdf["exact_match"].sum()), "total": int(len(cdf)),
            "exact_accuracy": round(cdf["exact_match"].mean()*100,1),
            "avg_word_f1": round(cdf["word_f1"].mean()*100,1),
            "avg_bleu": round(cdf["bleu"].mean()*100,1),
            "avg_rouge_l": round(cdf["rouge_l"].mean()*100,1),
        }
    return {
        "model": model_name, "num_samples": total,
        "exact_match_accuracy": round(em/total*100,1),
        "average_word_f1": round(np.mean(f1s)*100,1),
        "average_bleu": round(np.mean(bleus)*100,1),
        "average_rouge_l": round(np.mean(rouges)*100,1),
        "ece": round(ece*100,2),
        "exact_matches": em,
        "partial_matches_f1_gte_50": sum(1 for r in results if r["word_f1"]>=0.5),
        "total": total, "per_complexity": per_c,
    }

def print_eval_summary(s):
    print(f"\n{'='*70}")
    print(f"  {s.get('model','?').upper()} — EVALUATION SUMMARY")
    print(f"{'='*70}")
    print(f"  Accuracy (EM): {s.get('exact_matches',0)}/{s.get('total',0)} ({s.get('exact_match_accuracy',0):.1f}%)")
    print(f"  Word F1:       {s.get('average_word_f1',0):.1f}%")
    print(f"  BLEU:          {s.get('average_bleu',0):.1f}%")
    print(f"  ROUGE-L:       {s.get('average_rouge_l',0):.1f}%")
    print(f"  ECE:           {s.get('ece',0):.2f}%")
    print(f"  Per-Complexity:")
    for k,v in s.get('per_complexity',{}).items():
        print(f"    {k}: EM {v['exact_matches']}/{v['total']}, F1 {v['avg_word_f1']:.1f}%, BLEU {v['avg_bleu']:.1f}%, ROUGE-L {v['avg_rouge_l']:.1f}%")
    print(f"{'='*70}")

def save_model_results(results, key):
    s = compute_summary(results, key)
    pd.DataFrame(results).to_csv(os.path.join(RESULTS_DIR, f"{key}_predictions.csv"), index=False)
    with open(os.path.join(RESULTS_DIR, f"{key}_summary.json"), "w") as f:
        json.dump(s, f, indent=2)
    return s

print("Evaluation utilities loaded (Accuracy, F1, BLEU, ROUGE-L, ECE).")

## 4b. Prediction Visualization Grid

In [ ]:
def plot_prediction_grid(results, model_name, model_key, max_show=6):
    """Create a 2x3 grid of predictions overlaid on images."""
    show = results[:max_show]
    n = len(show)
    if n == 0: print("No results to visualize."); return
    cols = min(3, n); rows_n = math.ceil(n / cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(7*cols, 6*rows_n))
    if rows_n == 1 and cols == 1: axes = np.array([axes])
    axes = np.atleast_2d(axes)
    fig.suptitle(f"{model_name} Predictions on Kvasir-VQA-x1",
                 fontsize=16, fontweight='bold', y=1.01)
    for i, r in enumerate(show):
        row_i, col_i = divmod(i, cols)
        ax = axes[row_i][col_i]
        img_path = os.path.join(IMAGE_DIR, f"{r['img_id']}.jpg")
        if os.path.exists(img_path):
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "Image\nNot Found", ha='center', va='center',
                    fontsize=14, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        # Status color
        if r['exact_match']:
            status = 'EXACT MATCH ✓'; color = '#27ae60'
        elif r['word_f1'] >= 0.5:
            status = 'PARTIAL ~'; color = '#f39c12'
        else:
            status = 'WRONG ✗'; color = '#e74c3c'
        # Overlay text
        txt = (f"{status} | F1: {r['word_f1']:.2f} | Complexity {r['complexity']}\n"
               f"Q: {r['question'][:90]}\n"
               f"GT: {r['ground_truth'][:70]}\n"
               f"Pred: {r['prediction'][:70]}")
        ax.text(0.02, 0.98, txt, transform=ax.transAxes, fontsize=7,
                verticalalignment='top', color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.75))
    # Hide empty axes
    for i in range(n, rows_n * cols):
        r_i, c_i = divmod(i, cols)
        axes[r_i][c_i].axis('off')
    plt.tight_layout()
    save_path = os.path.join(RESULTS_DIR, f"{model_key}_prediction_grid.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Grid saved to {save_path}")

# Store results globally for grid visualization
all_results = {}
print("Prediction grid function loaded.")

## 5. Load Test Data

In [ ]:
test_df = pd.read_csv(os.path.join(DATA_DIR, "kvasir_vqa_x1_test.csv"))
sample_df = select_diverse_samples(test_df, NUM_SAMPLES)
print(f"Test: {len(test_df)} | Selected: {len(sample_df)}")
print(sample_df["complexity"].value_counts().sort_index().to_string())

## 6. Load Baseline Results

In [ ]:
all_summaries = {}
bp = os.path.join(RESULTS_DIR, "baseline_summary.json")
if os.path.exists(bp):
    with open(bp) as f: baseline_summary = json.load(f)
    baseline_summary["model"] = "BLIP-2 (Baseline)"
    # Add default values for new metrics if missing
    baseline_summary.setdefault("average_bleu", 0.0)
    baseline_summary.setdefault("average_rouge_l", 0.0)
    baseline_summary.setdefault("ece", 0.0)
else:
    baseline_summary = {"model": "BLIP-2 (Baseline)", "num_samples": 18,
        "exact_match_accuracy": 0.0, "average_word_f1": 27.0,
        "average_bleu": 0.0, "average_rouge_l": 0.0, "ece": 0.0,
        "exact_matches": 0, "partial_matches_f1_gte_50": 1, "total": 18,
        "per_complexity": {
            "level_1": {"exact_matches":0,"total":6,"exact_accuracy":0.0,"avg_word_f1":18.9,"avg_bleu":0.0,"avg_rouge_l":0.0},
            "level_2": {"exact_matches":0,"total":6,"exact_accuracy":0.0,"avg_word_f1":27.0,"avg_bleu":0.0,"avg_rouge_l":0.0},
            "level_3": {"exact_matches":0,"total":6,"exact_accuracy":0.0,"avg_word_f1":35.1,"avg_bleu":0.0,"avg_rouge_l":0.0}}}
all_summaries["BLIP-2 (Baseline)"] = baseline_summary
print_eval_summary(baseline_summary)

---
## 7. Generic Model Evaluation Function

In [ ]:
def run_model_evaluation(model, processor, model_name, model_key,
                         is_instructblip=False, is_florence2=False, is_text_only=False):
    results = []
    print(f"\n[INFO] Running {model_name} on {len(sample_df)} samples...\n")
    for idx, (_, row) in enumerate(sample_df.iterrows()):
        img_path = os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg")
        if not os.path.exists(img_path):
            print(f"[SKIP] Image not found: {row['img_id']}")
            continue
        image = Image.open(img_path).convert("RGB")
        question = row["question"]
        dtype = torch.float16 if DEVICE == "cuda" else torch.float32
        try:
            if is_florence2:
                prompt = f"<VQA> {question}"
                inputs = processor(text=prompt, images=image, return_tensors="pt")
                inputs = {k: v.to(DEVICE, dtype) if v.is_floating_point()
                          else v.to(DEVICE) for k, v in inputs.items()}
            elif is_instructblip:
                prompt = f"Question: {question} Answer:"
                inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE, dtype)
            elif is_text_only:
                prompt = f"Based on the medical image, answer: {question} Answer:"
                inputs = processor(prompt, return_tensors="pt", truncation=True, max_length=512)
                inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            else:
                prompt = f"Question: {question} Answer:"
                try:
                    inputs = processor(images=image, text=prompt, return_tensors="pt")
                    inputs = {k: v.to(DEVICE, dtype) if v.is_floating_point()
                              else v.to(DEVICE) for k, v in inputs.items()}
                except Exception:
                    inputs = processor(prompt, return_tensors="pt", truncation=True, max_length=512)
                    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                        do_sample=False, num_beams=3)
            if is_instructblip or is_florence2:
                prediction = processor.decode(outputs[0], skip_special_tokens=True).strip()
                if is_florence2 and prediction.startswith('<VQA>'):
                    prediction = prediction[5:].strip()
            else:
                input_len = inputs.get("input_ids", torch.tensor([[]])).shape[-1]
                prediction = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        except Exception as e:
            print(f"[ERROR] Sample {idx}: {e}")
            prediction = ""
        result = evaluate_single(row, prediction)
        results.append(result)
        status = "✓" if result["exact_match"] else "~" if result["word_f1"]>=0.5 else "✗"
        print(f"[{idx+1}/{len(sample_df)}] {status} F1:{result['word_f1']:.2f} BLEU:{result['bleu']:.2f} RL:{result['rouge_l']:.2f}")
        print(f"  Q: {result['question'][:70]}")
        print(f"  GT: {result['ground_truth'][:70]} | Pred: {result['prediction'][:70]}")

    summary = save_model_results(results, model_key)
    print_eval_summary(summary)
    # Store for grid visualization
    all_results[model_key] = results
    # Show prediction grid
    plot_prediction_grid(results, model_name, model_key)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return summary

print("Ready.")

---
## 8. Model 1: MoE-TinyMed

Medical MoE architecture. HF: `Joycean0301/MoE-Tinymed-phi2-llava`

In [ ]:
MOE_IDS = ["Joycean0301/MoE-Tinymed-phi2-llava", "JsST/TinyMed"]
moe_model = moe_proc = None
for mid in MOE_IDS:
    try:
        print(f"[INFO] Loading MoE-TinyMed from: {mid}")
        try: moe_proc = AutoProcessor.from_pretrained(mid, trust_remote_code=True)
        except Exception: moe_proc = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
        if moe_proc.pad_token is None: moe_proc.pad_token = moe_proc.eos_token
        moe_model = AutoModelForCausalLM.from_pretrained(mid,
            torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
            trust_remote_code=True, device_map="auto" if DEVICE=="cuda" else None)
        moe_model.eval()
        print(f"[INFO] Loaded: {mid}"); break
    except Exception as e: print(f"[WARN] {e}"); continue
if moe_model is None: print("[ERROR] MoE-TinyMed failed.")

In [ ]:
if moe_model is not None:
    all_summaries["MoE-TinyMed"] = run_model_evaluation(
        moe_model, moe_proc, "MoE-TinyMed", "moe_tinymed")
else: print("[SKIP] MoE-TinyMed")

---
## 9. Model 2: BiomedGPT (Text-only)

HF: `PharMolix/BioMedGPT-LM-7B` — text-only biomedical LLM.

In [ ]:
biomed_model = biomed_proc = None
try:
    mid = "PharMolix/BioMedGPT-LM-7B"
    print(f"[INFO] Loading BiomedGPT from: {mid}")
    biomed_proc = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    if biomed_proc.pad_token is None:
        biomed_proc.pad_token = biomed_proc.eos_token
        print("[INFO] Set pad_token = eos_token")
    biomed_model = AutoModelForCausalLM.from_pretrained(mid,
        torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
        trust_remote_code=True, device_map="auto" if DEVICE=="cuda" else None)
    biomed_model.eval()
    print("[INFO] BiomedGPT loaded.")
except Exception as e: print(f"[ERROR] {e}")

In [ ]:
if biomed_model is not None:
    all_summaries["BiomedGPT"] = run_model_evaluation(
        biomed_model, biomed_proc, "BiomedGPT", "biomedgpt", is_text_only=True)
else: print("[SKIP] BiomedGPT")

---
## 10. Model 3: Florence-2 (replaces TinyGPT-V)

0.23B VLM. HF: `microsoft/Florence-2-base`

In [ ]:
fl_model = fl_proc = None
for mid in ["microsoft/Florence-2-base", "microsoft/Florence-2-large"]:
    try:
        print(f"[INFO] Loading Florence-2 from: {mid}")
        fl_proc = AutoProcessor.from_pretrained(mid, trust_remote_code=True)
        fl_model = AutoModelForCausalLM.from_pretrained(mid,
            torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
            trust_remote_code=True)
        if DEVICE=="cuda": fl_model = fl_model.to(DEVICE)
        fl_model.eval()
        print(f"[INFO] Loaded: {mid}"); break
    except Exception as e: print(f"[WARN] {e}"); continue
if fl_model is None: print("[ERROR] Florence-2 failed.")

In [ ]:
if fl_model is not None:
    all_summaries["Florence-2"] = run_model_evaluation(
        fl_model, fl_proc, "Florence-2", "florence2", is_florence2=True)
else: print("[SKIP] Florence-2")

---
## 11. Model 4: MiniGPT-4 (InstructBLIP-Vicuna)

~7B. HF: `Salesforce/instructblip-vicuna-7b`

In [ ]:
mini_model = mini_proc = None
for mid in ["Salesforce/instructblip-vicuna-7b", "Salesforce/instructblip-flan-t5-xl"]:
    try:
        print(f"[INFO] Loading MiniGPT-4 from: {mid}")
        mini_proc = InstructBlipProcessor.from_pretrained(mid)
        mini_model = InstructBlipForConditionalGeneration.from_pretrained(mid,
            torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
            device_map="auto" if DEVICE=="cuda" else None)
        mini_model.eval()
        print(f"[INFO] Loaded: {mid}"); break
    except Exception as e: print(f"[WARN] {e}"); continue
if mini_model is None: print("[ERROR] MiniGPT-4 failed.")

In [ ]:
if mini_model is not None:
    all_summaries["MiniGPT-4"] = run_model_evaluation(
        mini_model, mini_proc, "MiniGPT-4", "minigpt4", is_instructblip=True)
else: print("[SKIP] MiniGPT-4")

---
## 12. Full Comparison Table (All Metrics)

In [ ]:
params_map = {"BLIP-2 (Baseline)": "~2.7B", "MoE-TinyMed": "~3.6B",
              "BiomedGPT": "7B", "Florence-2": "0.23B", "MiniGPT-4": "~7B"}
domain_map = {"BLIP-2 (Baseline)": "General", "MoE-TinyMed": "Medical",
              "BiomedGPT": "Biomedical", "Florence-2": "General", "MiniGPT-4": "General"}

rows = []
for name, s in all_summaries.items():
    r = {"Model": name, "Params": params_map.get(name,"?"), "Domain": domain_map.get(name,"?"),
         "Accuracy (%)": s.get("exact_match_accuracy",0),
         "F1 (%)": s.get("average_word_f1",0),
         "BLEU (%)": s.get("average_bleu",0),
         "ROUGE-L (%)": s.get("average_rouge_l",0),
         "ECE (%)": s.get("ece",0),
         "Samples": s.get("total", s.get("num_samples",0))}
    rows.append(r)

comp_df = pd.DataFrame(rows).sort_values("F1 (%)", ascending=False)
print("\n" + "="*110 + "\n  VSLM COMPARISON — ALL METRICS\n" + "="*110)
print(comp_df.to_string(index=False))
print("="*110)
comp_df.to_csv(os.path.join(RESULTS_DIR, "vslm_comparison.csv"), index=False)
print(f"\nSaved to {os.path.join(RESULTS_DIR, 'vslm_comparison.csv')}")

## 13. Visualization — Bar Charts

In [ ]:
from matplotlib.patches import Patch

metrics = [('Accuracy (%)', 'Exact Match Accuracy'),
           ('F1 (%)', 'Word F1 Score'),
           ('BLEU (%)', 'BLEU Score'),
           ('ROUGE-L (%)', 'ROUGE-L Score'),
           ('ECE (%)', 'Expected Calibration Error')]

fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 5))
models = comp_df["Model"].tolist()
colors = ['#e74c3c' if 'Baseline' in m else '#3498db' if 'Med' in m or 'Biomed' in m
          else '#2ecc71' for m in models]

for ax, (col, title) in zip(axes, metrics):
    vals = comp_df[col].tolist()
    bars = ax.barh(models, vals, color=colors, edgecolor='white')
    ax.set_xlabel(col); ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlim(0, max(max(vals)*1.3, 5))
    for b,v in zip(bars,vals):
        ax.text(b.get_width()+0.3, b.get_y()+b.get_height()/2, f'{v:.1f}', va='center', fontsize=8)

plt.suptitle('VSLM Performance Comparison on Kvasir-VQA-x1', fontsize=14, fontweight='bold', y=1.02)
fig.legend(handles=[Patch(fc='#e74c3c',label='Baseline'), Patch(fc='#3498db',label='Medical/Biomed'),
    Patch(fc='#2ecc71',label='General')], loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.06))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'vslm_all_metrics_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Per-Complexity Breakdown

In [ ]:
# Per-complexity F1, BLEU, ROUGE-L breakdown
pc_rows = []
for name, s in all_summaries.items():
    for k, v in s.get('per_complexity', {}).items():
        pc_rows.append({
            'Model': name, 'Level': k.replace('level_',''),
            'EM (%)': v.get('exact_accuracy', 0),
            'F1 (%)': v.get('avg_word_f1', 0),
            'BLEU (%)': v.get('avg_bleu', 0),
            'ROUGE-L (%)': v.get('avg_rouge_l', 0),
        })
if pc_rows:
    pc_df = pd.DataFrame(pc_rows)
    print("\nPer-Complexity Breakdown:")
    print(pc_df.to_string(index=False))
    # Grouped bar chart for F1 by complexity
    levels = sorted(pc_df['Level'].unique())
    model_names = pc_df['Model'].unique()
    x = np.arange(len(levels))
    width = 0.8 / len(model_names)
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, mn in enumerate(model_names):
        vals = [pc_df[(pc_df['Model']==mn) & (pc_df['Level']==l)]['F1 (%)'].values[0]
                if len(pc_df[(pc_df['Model']==mn) & (pc_df['Level']==l)]) > 0 else 0 for l in levels]
        ax.bar(x + i*width, vals, width, label=mn)
    ax.set_xlabel('Complexity Level'); ax.set_ylabel('Avg Word F1 (%)')
    ax.set_title('Word F1 by Complexity Level', fontweight='bold')
    ax.set_xticks(x + width*(len(model_names)-1)/2)
    ax.set_xticklabels([f'Level {l}' for l in levels])
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'vslm_per_complexity.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 15. Conclusion

In [ ]:
bl = all_summaries.get("BLIP-2 (Baseline)", {})
bl_f1 = bl.get("average_word_f1",27.0)
bl_em = bl.get("exact_match_accuracy",0.0)
bl_bleu = bl.get("average_bleu",0.0)
bl_rl = bl.get("average_rouge_l",0.0)
print(f"\n{'='*80}")
print(f"  VERDICT: VSLMs vs BLIP-2 Baseline")
print(f"{'='*80}")
print(f"  Baseline: Acc={bl_em:.1f}% | F1={bl_f1:.1f}% | BLEU={bl_bleu:.1f}% | ROUGE-L={bl_rl:.1f}%\n")
better = []
for name, s in all_summaries.items():
    if 'Baseline' in name: continue
    em = s.get('exact_match_accuracy',0)
    f1 = s.get('average_word_f1',0)
    bleu = s.get('average_bleu',0)
    rl = s.get('average_rouge_l',0)
    ece = s.get('ece',0)
    print(f"  {name}:")
    print(f"    Acc={em:.1f}% ({em-bl_em:+.1f}pp) | F1={f1:.1f}% ({f1-bl_f1:+.1f}pp)")
    print(f"    BLEU={bleu:.1f}% ({bleu-bl_bleu:+.1f}pp) | ROUGE-L={rl:.1f}% ({rl-bl_rl:+.1f}pp) | ECE={ece:.2f}%")
    if f1 > bl_f1 or em > bl_em: better.append(name)
print()
if better:
    print(f"  ✓ Beat baseline: {', '.join(better)}")
else:
    print(f"  ✗ No VSLM outperformed baseline in zero-shot. Fine-tuning recommended.")
print(f"{'='*80}")
with open(os.path.join(RESULTS_DIR, 'vslm_final_comparison.json'), 'w') as f:
    json.dump(all_summaries, f, indent=2)
print(f"\nResults saved to {RESULTS_DIR}")